In [26]:
import pandas as pd
import numpy as np
# Carga del dataset transaccional
df_ventas = pd.read_csv('Raw/ventas_order_line.csv')

# Inspección rápida
print("--- Muestra de Ventas (Order Lines) ---")
display(df_ventas.head())

print("\n--- Información Estructural ---")
df_ventas.info()

--- Muestra de Ventas (Order Lines) ---


,id_linea,id_orden,id_producto,cantidad,precio_unitario_s_iva,iva,monto_linea
0,5629,1253,104,6,3630.49,0.105,21782.94
1,5171,1069,105,15,9203.61,0.210,138054.15
2,5842,1336,105,16,9390.03,0.210,150240.48
3,5892,1355,117,2,5295.04,0.210,10590.08
4,5862,1344,109,19,6766.85,0.210,128570.15



--- Información Estructural ---
<class 'pandas.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_linea               1096 non-null   int64  
 1   id_orden               1096 non-null   int64  
 2   id_producto            1096 non-null   int64  
 3   cantidad               1096 non-null   int64  
 4   precio_unitario_s_iva  1096 non-null   float64
 5   iva                    1096 non-null   float64
 6   monto_linea            1096 non-null   float64
dtypes: float64(3), int64(4)
memory usage: 60.1 KB


In [27]:
# 1. Conversión de Identificadores
# Convertimos las IDs de numérico a texto para que Pandas no intente sumarlas o promediarlas
cols_ids = ['id_linea', 'id_orden', 'id_producto']
for col in cols_ids:
    df_ventas[col] = df_ventas[col].astype(str)

# 2. Auditoría y resolución de duplicados en la clave primaria (id_linea)
duplicados_linea = df_ventas.duplicated(subset=['id_linea']).sum()
print(f"Alerta de integridad: Se encontraron {duplicados_linea} líneas de venta duplicadas.")

if duplicados_linea > 0:
    # Nos quedamos con la primera aparición y descartamos los clones para no inflar la facturación
    df_ventas = df_ventas.drop_duplicates(subset=['id_linea'], keep='first')
    print("Los duplicados han sido eliminados. Tabla de ventas purgada.")

Alerta de integridad: Se encontraron 38 líneas de venta duplicadas.
Los duplicados han sido eliminados. Tabla de ventas purgada.


In [28]:
# 1. Corrección de valores negativos (Transformación a Valor Absoluto)
# Pasamos el precio y el monto a positivo usando .abs()
df_ventas['precio_unitario_s_iva'] = df_ventas['precio_unitario_s_iva'].abs()
df_ventas['monto_linea'] = df_ventas['monto_linea'].abs()

# 2. Verificación de cálculo base (Auditoría del sistema origen)
# Revisamos que cantidad * precio_unitario sea igual a monto_linea
monto_teorico = (df_ventas['cantidad'] * df_ventas['precio_unitario_s_iva']).round(2)
cuadre_matematico = (monto_teorico == df_ventas['monto_linea'].round(2)).all()
print(f"¿La matemática base del sistema origen es correcta?: {'Sí' if cuadre_matematico else 'No'}")

# 3. Creación de nuevas métricas financieras (Feature Engineering)
# El monto actual es "Sin IVA" (Subtotal). Agregamos el monto total a facturar (Con IVA)
df_ventas['monto_total_con_iva'] = df_ventas['monto_linea'] * (1 + df_ventas['iva'])
df_ventas['monto_total_con_iva'] = df_ventas['monto_total_con_iva'].round(2)

# Agregamos la columna que representa solo el valor del impuesto recaudado
df_ventas['monto_impuesto'] = df_ventas['monto_total_con_iva'] - df_ventas['monto_linea']
df_ventas['monto_impuesto'] = df_ventas['monto_impuesto'].round(2)

# 4. Resumen financiero final
print("\n--- Resumen Estadístico de Facturación (Corregido) ---")
display(df_ventas[['cantidad', 'precio_unitario_s_iva', 'monto_linea', 'monto_total_con_iva']].describe().round(2))

¿La matemática base del sistema origen es correcta?: Sí

--- Resumen Estadístico de Facturación (Corregido) ---


,cantidad,precio_unitario_s_iva,monto_linea,monto_total_con_iva
count,1058.00,1058.00,1058.00,1058.00
mean,10.50,6100.30,64060.38,74398.55
std,5.72,2595.20,47658.12,55670.59
min,1.00,2439.03,2459.68,2976.21
25%,6.00,3615.03,28215.31,32514.97
50%,11.00,5408.82,51133.68,60039.62
75%,15.00,8262.37,89087.32,103219.08
max,20.00,11617.43,226736.40,270411.53


In [29]:
df_ventas

,id_linea,id_orden,id_producto,cantidad,precio_unitario_s_iva,iva,monto_linea,monto_total_con_iva,monto_impuesto
0,5629,1253,104,6,3630.49,0.105,21782.94,24070.15,2287.21
1,5171,1069,105,15,9203.61,0.210,138054.15,167045.52,28991.37
2,5842,1336,105,16,9390.03,0.210,150240.48,181790.98,31550.50
3,5892,1355,117,2,5295.04,0.210,10590.08,12814.00,2223.92
4,5862,1344,109,19,6766.85,0.210,128570.15,155569.88,26999.73
...,...,...,...,...,...,...,...,...,...
1089,5123,1050,117,16,5224.27,0.210,83588.32,101141.87,17553.55
1090,5858,1342,108,20,8260.05,0.105,165201.00,182547.10,17346.10
1092,6035,1412,117,14,5267.98,0.210,73751.72,89239.58,15487.86
1094,5131,1053,120,17,8368.53,0.105,142265.01,157202.84,14937.83


In [30]:
display(df_ventas.dtypes)

id_linea                     str
id_orden                     str
id_producto                  str
cantidad                   int64
precio_unitario_s_iva    float64
iva                      float64
monto_linea              float64
monto_total_con_iva      float64
monto_impuesto           float64
dtype: object

In [31]:
# Al menos asegurate de que los tipos estén bien antes de exportar
df_ventas = df_ventas.astype({
    'id_linea': 'string',
    'id_orden': 'string',
    'id_producto': 'string',
    'cantidad': 'int64',
    'precio_unitario_s_iva': 'float64',
    'iva': 'float64',
    'monto_linea': 'float64',
    'monto_total_con_iva': 'float64',
    'monto_impuesto': 'float64'
})

df_ventas.to_csv('ventas.csv', index=False)

In [32]:
# Exportar el DataFrame de las líneas de órdenes de venta
df_ventas.to_csv('Processed/ventas_order_line_procesado.csv', index=False, encoding='utf-8')

print("✅ df_ventas exportado exitosamente a Processed/ventas_order_line_procesado.csv")

✅ df_ventas exportado exitosamente a Processed/ventas_order_line_procesado.csv
